# Simple version: model with 11 extra word-family features
Same as `model.ipynb`, but reading `data/features_cole_simple.csv`: the original 36 features plus 11 word-family features
made by `src/cole_features_simple.py`.

| Group | Feature (rate per 100 words) | Leans |
|---|---|---|
| Spoken vs. written style | discourse markers (oh, yeah, well, okay, I mean) | human |
| | casual verbs (got, gonna, trying, come on) | human |
| | slang (gonna, bro, dude, guys, nah) | human |
| | subordinators (who, while, before, because) | AI |
| | time connectives (then, later, finally, one day) | AI |
| Story content | reveal words (truth, secret, hiding, explain, knew) | AI |
| | absolutist words (never, everything, forever, nothing) | AI |
| | kinship (mother, father, sister, husband) | AI |
| | future modals (will, won't, going to, promise) | AI |
| | love talk (I love you, my love, honey) | AI |
| | apologies (sorry, my fault, forgive me) | human |


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df=pd.read_csv("../data/features_cole_simple.csv")  # features.csv + 11 word-family features (src/cole_features_simple.py)

In [3]:
df.head(2)

                                          transcript  ...  y
0  First day at the new rich kid's school. I hope...  ...  0
1  I said, Mom, never let me call her mom. Not on...  ...  1

[2 rows x 49 columns]

In [4]:
from sklearn.ensemble import RandomForestClassifier

# Separate features and target
X = df.drop(columns=["label", "transcript", "y"], errors="ignore")
y = df["y"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 1. Evaluate Scaled Logistic Regression
log_model = LogisticRegression(max_iter=500)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

print("--- Logistic Regression (Scaled) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print(classification_report(y_test, y_pred_log))

# 2. Evaluate Random Forest (Handles non-linear relationships and scales automatically)
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("\n--- Random Forest Classifier ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf))

--- Logistic Regression (Scaled) ---
Accuracy: 0.7778
              precision    recall  f1-score   support

           0       0.82      0.72      0.77        32
           1       0.74      0.84      0.79        31

    accuracy                           0.78        63
   macro avg       0.78      0.78      0.78        63
weighted avg       0.78      0.78      0.78        63


--- Random Forest Classifier ---
Accuracy: 0.7143
              precision    recall  f1-score   support

           0       0.77      0.62      0.69        32
           1       0.68      0.81      0.74        31

    accuracy                           0.71        63
   macro avg       0.72      0.72      0.71        63
weighted avg       0.72      0.71      0.71        63



## Do the extra features help? (fair comparison)
One 80/20 split is only 63 test clips, so a single score swings by several points depending on the split.
Below, both feature sets get the **same 20 train/test splits** (5-fold CV repeated 4 times).
For logistic regression the features are scaled and the regularization strength `C` is tuned **inside each training
fold only**, so the test clips never influence any choice.

In [5]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

EXTRA = [c for c in X.columns if c.endswith("_per100") and c in [
    "discourse_markers_per100", "casual_verbs_per100", "slang_per100", "subordinators_per100", "time_connectives_per100",
    "reveal_words_per100", "absolutist_per100", "kinship_per100", "future_modals_per100", "love_talk_per100", "apology_per100"]]
ORIGINAL = [c for c in X.columns if c not in EXTRA]

def tuned_lr():
    return GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
                        {"logisticregression__C": [0.01, 0.03, 0.1, 0.3, 1, 3]},
                        cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc")

splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=42).split(X, y))
results = {}
for name, cols in [("original 36", ORIGINAL), ("original 36 + extra 11", ORIGINAL + EXTRA)]:
    lr_acc, lr_auc, rf_acc = [], [], []
    for tr, te in splits:
        m = tuned_lr().fit(X.iloc[tr][cols], y.iloc[tr])
        lr_acc.append(accuracy_score(y.iloc[te], m.predict(X.iloc[te][cols])))
        lr_auc.append(roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te][cols])[:, 1]))
        rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(X.iloc[tr][cols], y.iloc[tr])
        rf_acc.append(accuracy_score(y.iloc[te], rf.predict(X.iloc[te][cols])))
    results[name] = np.array(lr_acc), np.array(lr_auc), np.array(rf_acc)
    print(f"{name:24s}  LogReg accuracy {np.mean(lr_acc):.3f}  AUC {np.mean(lr_auc):.3f}  |  RandomForest accuracy {np.mean(rf_acc):.3f}")

a0, u0, r0 = results["original 36"]; a1, u1, r1 = results["original 36 + extra 11"]
print(f"\nWith the extra features: LogReg better on {np.mean(a1 > a0):.0%} of splits (worse on {np.mean(a1 < a0):.0%}), "
      f"AUC better on {np.mean(u1 > u0):.0%};  RandomForest better on {np.mean(r1 > r0):.0%} (worse on {np.mean(r1 < r0):.0%})")

original 36               LogReg accuracy 0.759  AUC 0.825  |  RandomForest accuracy 0.735
original 36 + extra 11    LogReg accuracy 0.812  AUC 0.873  |  RandomForest accuracy 0.786

With the extra features: LogReg better on 90% of splits (worse on 10%), AUC better on 95%;  RandomForest better on 90% (worse on 10%)


## Which extra features does the model lean on?
Standardized logistic-regression coefficients (C=0.03) fitted on all clips: positive = more AI-like.

In [6]:
final = make_pipeline(StandardScaler(), LogisticRegression(C=0.03, max_iter=5000)).fit(X, y)
coefs = pd.Series(final[-1].coef_[0], index=X.columns)
print(coefs[EXTRA].sort_values().round(3).to_string())

discourse_markers_per100   -0.416
casual_verbs_per100        -0.321
slang_per100               -0.208
apology_per100             -0.197
time_connectives_per100     0.116
kinship_per100              0.122
love_talk_per100            0.122
absolutist_per100           0.132
subordinators_per100        0.145
future_modals_per100        0.197
reveal_words_per100         0.223
